# 🎛️ Filters / Kernels — Notes + Interview
---
> **Simple English** | **Interview Ready**

## 📌 What are Filters/Kernels? (Simple English)
- A **filter (kernel)** is a small matrix of learnable numbers (weights)
- It slides over the image to detect a **specific feature** (edge, curve, color)
- Filter size is usually **3×3** or **5×5** — small but powerful
- **Multiple filters** → detect multiple features simultaneously
- Each filter produces **1 feature map** → 32 filters → 32 feature maps
- CNN **learns** the best filter values during training — not hand-crafted!

## 🔑 Key Facts
| Property | Detail |
|---|---|
| Size | Usually 3×3 (most common), 5×5, 7×7 |
| Depth | Must match input channels (RGB → depth=3) |
| Number | Hyperparameter you choose (32, 64, 128...) |
| Values | Learned by backpropagation |
| Output | 1 filter → 1 feature map |

## 🧱 Filter Examples
| Filter Type | What it Detects |
|---|---|
| Sobel horizontal | Horizontal edges |
| Sobel vertical | Vertical edges |
| Gaussian blur | Smooth/blur |
| Sharpen | Sharp edges |
| CNN learned filters | Whatever helps minimize loss |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Famous hand-crafted filters (before CNNs learned them)
filters = {
    'Sobel Horizontal\n(detects H edges)': np.array([[-1,-2,-1],[0,0,0],[1,2,1]]),
    'Sobel Vertical\n(detects V edges)' : np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
    'Gaussian Blur'                       : np.array([[1,2,1],[2,4,2],[1,2,1]])/16,
    'Sharpen'                             : np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]),
    'Edge All Dir.'                       : np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]]),
}

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (name, f) in zip(axes, filters.items()):
    im = ax.imshow(f, cmap='RdBu', vmin=-2, vmax=2)
    ax.set_title(name, fontsize=9)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{f[i,j]:.2f}', ha='center', va='center', fontsize=9, fontweight='bold')
    ax.axis('off')
plt.suptitle('Common 3×3 Filters (CNNs LEARN these automatically)', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Apply multiple filters to an image
import numpy as np
import matplotlib.pyplot as plt

def conv2d(image, kernel):
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh, ow = ih-kh+1, iw-kw+1
    out = np.zeros((oh,ow))
    for i in range(oh):
        for j in range(ow):
            out[i,j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return out

# Simple image
img = np.array([
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0],
    [0,0,255,255,255,0,0],
    [0,0,255,255,255,0,0],
    [0,0,255,255,255,0,0],
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0],
], dtype=float)

filters_list = {
    'Original':          None,
    'Sobel H (edges↔)':  np.array([[-1,-2,-1],[0,0,0],[1,2,1]]),
    'Sobel V (edges↕)':  np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
    'All Edges':         np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]]),
}

fig, axes = plt.subplots(1, 4, figsize=(14,4))
for ax, (name, filt) in zip(axes, filters_list.items()):
    result = img if filt is None else conv2d(img, filt)
    ax.imshow(result, cmap='gray')
    ax.set_title(name)
    ax.axis('off')
plt.suptitle('Same image, different filters → different features extracted')
plt.tight_layout(); plt.show()

In [ ]:
# CNN with 32 filters — see learned filter shapes
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='softmax')
])
model.summary()
print(f"\nLayer 1: 32 filters of size 3×3×1 = {32*3*3*1} weights + {32} biases = {32*3*3*1+32} params")
print(f"Layer 2: 64 filters of size 3×3×32= {64*3*3*32} weights + {64} biases = {64*3*3*32+64} params")
print("\nEach filter is LEARNED automatically during training!")

## 🗣️ Interview Q&A

**Q: What is a filter/kernel in CNN?**
> A small learnable weight matrix (e.g. 3×3) that slides over input to detect specific features. CNN learns optimal filter values through backpropagation.

**Q: Why use 3×3 filters instead of larger ones?**
> Two 3×3 filters stacked = same receptive field as one 5×5 but with fewer parameters (2×9=18 vs 25) and an extra non-linearity. VGG showed 3×3 is optimal.

**Q: How many parameters in a Conv layer?**
> `(kernel_H × kernel_W × input_channels + 1) × num_filters`
> E.g. 3×3×3 RGB input with 32 filters = (3×3×3+1)×32 = 896 params

**Q: What is the depth of a filter?**
> Must match input channels. Grayscale → depth=1, RGB → depth=3, previous conv output with 64 filters → depth=64.